# Clase 3 — NLP aplicado: tokenización, embeddings, clasificación y extracción de entidades

## Pregunta central

> **¿Cómo pasa un programa de recibir una frase a reconocer de qué trata y sacar sus datos concretos?**

## Idea principal

Un programa no recibe una oración y la comprende como una persona. Primero la divide en piezas, después puede representar su sentido con números, asignarle una etiqueta o localizar datos concretos dentro del texto. En este notebook vas a recorrer esas cuatro operaciones paso a paso y con herramientas simples.

## Objetivos de aprendizaje

Al finalizar la clase deberías poder:

- Explicar por qué un modelo trabaja con **tokens** y no con el texto “de una”.
- Interpretar un **embedding** como una ubicación numérica del significado aproximado.
- Distinguir **clasificación** de **extracción de entidades**.
- Reconocer cuándo usar `tiktoken`, `sentence-transformers` o `scikit-learn`.
- Construir un extractor pequeño con listas de términos y reglas visibles.
- Detectar qué casos resuelve una regla y cuáles quedan afuera.

## Recorrido de la clase

| Mitad | Qué vas a recorrer | Resultado esperado |
|---|---|---|
| 1 — Teoría con demos breves | Tokens, embeddings, clasificación y entidades | Entender qué hace cada operación y conocer herramientas disponibles |
| 2 — Práctica | Un extractor simplificado para mensajes de consultorio | Modificar listas controladas, ejecutar y observar aciertos y ausencias |

## Cómo trabajar con este notebook

1. Ejecutá las celdas en el orden propuesto.
2. Antes de cambiar algo, observá la entrada y la salida.
3. En la práctica, tocá solamente las variables marcadas con `TODO`.
4. Volvé a ejecutar desde la celda que modificaste para ver el efecto.
5. Todo el material usa mensajes sintéticos inventados para aprender.

## Glosario mínimo

| Término | Explicación humana |
|---|---|
| Token | Una pieza de texto que el modelo puede convertir en un número |
| Embedding | Una lista de números que resume “de qué se trata” un texto |
| Vector | Una lista ordenada de números, como coordenadas |
| Clasificación | Elegir una etiqueta para un texto completo |
| Etiqueta | Nombre de una categoría, por ejemplo `turno` o `receta` |
| Entidad | Dato concreto localizado dentro de un texto |
| Gazetteer / lista de términos | Diccionario controlado de expresiones que queremos reconocer |
| TF-IDF | Forma de dar más peso a palabras útiles para distinguir textos |

---
## Un mapa antes de empezar

Las cuatro ideas responden preguntas distintas:

```text
“Necesito turno con cardiología el martes”
                     |
          +----------+-----------+
          |          |           |
          v          v           v
       tokens     embedding   clasificación
    [piezas]     [números]       “turno”
                                  |
                                  v
                          extracción de entidades
                     especialidad: cardiología
                     fecha: martes
```

| Si querés saber... | Operación |
|---|---|
| ¿En qué piezas se transforma la frase? | Tokenización |
| ¿Qué textos tienen un sentido parecido? | Embeddings |
| ¿Qué categoría describe al mensaje completo? | Clasificación |
| ¿Qué datos concretos aparecen adentro? | Extracción de entidades |

No son cuatro nombres para lo mismo. Son herramientas que miran el texto desde ángulos diferentes.

---
## 1. Tokenización: partir el texto en piezas manejables

Un modelo trabaja con números. Por eso el texto necesita una puerta de entrada: el **tokenizador** divide la cadena en piezas y asigna un identificador numérico a cada una.

> **Analogía del ticket de guardarropa.** Entregás una prenda y recibís un número. El número no “significa” la prenda por sí solo: sirve para identificarla dentro de un sistema. Un tokenizador hace algo parecido con piezas de texto.

```text
texto                    tokens                  identificadores
“cardiología mañana” -> [“card”, “iología”,     -> [1234, 5678, 901]
                          “ mañana”]
```

El corte real depende del tokenizador. No supongas que un token siempre es una palabra.

### Palabra frente a subpalabra

| Estrategia | Corte ilustrativo | Ventaja | Dificultad |
|---|---|---|---|
| Por palabra | `cardiología` | Fácil de leer | Palabras nuevas generan un vocabulario enorme |
| Por subpalabra | `card` + `iología` | Reutiliza piezas conocidas | El corte puede resultar raro para una persona |

Las subpalabras ayudan a manejar nombres, variantes, tildes y palabras poco frecuentes sin guardar cada palabra posible en un diccionario gigante.

### ¿Por qué el texto no entra “de una”? 

Una computadora puede guardar caracteres, pero el modelo necesita una secuencia numérica compatible con su vocabulario.

```text
frase escrita
     |
     v
tokenizador ---- usa su vocabulario ----+
     |                                   |
     v                                   v
lista de tokens                    lista de IDs
     |                                   |
     +----------------+------------------+
                      v
                    modelo
```

Tres ideas para retener:

1. **Token no equivale siempre a palabra.** Puede ser una palabra, parte de una palabra o un signo.
2. **Cada tokenizador corta a su manera.** Los IDs solo tienen sentido con el vocabulario que los creó.
3. **La cantidad de tokens importa.** Una frase corta suele usar pocos; una palabra inusual puede dividirse en varias piezas.

In [ ]:
# --- Demo breve: mirar tokens con tiktoken ---
import tiktoken

codificador = tiktoken.get_encoding("cl100k_base")
frases = [
    "Necesito un turno mañana.",
    "Cardiología",
    "electrocardiograma",
]

for frase in frases:
    ids = codificador.encode(frase)
    piezas = []
    for token_id in ids:
        pieza = codificador.decode([token_id])
        piezas.append(pieza)

    print("Texto:", frase)
    print("Piezas:", piezas)
    print("IDs:", ids)
    print("Cantidad:", len(ids))
    print("-" * 60)

### Cómo leer la demo

Mirá primero las **piezas**, no memorices los IDs. Probablemente una palabra habitual use menos piezas que una palabra larga o poco común.

`tiktoken` sirve para:

- observar cómo se parte un texto;
- contar tokens antes de procesarlo;
- convertir tokens a IDs y volver a texto.

> **Ojo:** un ID grande no significa que el token sea “más importante”. Es solamente su número de ficha dentro de ese vocabulario.

---
## 2. Embeddings: convertir el sentido en coordenadas

Un **embedding** representa un texto con una lista de números. La lista suele tener muchas posiciones, pero podés imaginarla como una ubicación en un mapa: textos parecidos quedan cerca y textos diferentes, más lejos.

> **Analogía del mapa de una biblioteca.** Los libros de cocina quedan cerca entre sí; los de electricidad ocupan otro sector. El mapa no copia cada página: ubica cada libro según su tema. Un embedding hace algo parecido con textos.

```text
                   mapa imaginario de significados

      “dolor de cabeza”  •────•  “me duele la cabeza”


                                  •  “quiero cambiar el turno”

         textos parecidos: cerca       tema distinto: lejos
```

El vector no es una traducción palabra por palabra. Es una representación numérica útil para comparar.

### Cercanía entre vectores

Supongamos, solo para dibujarlo, que cada embedding tuviera dos números:

| Texto | Vector inventado |
|---|---|
| “tengo fiebre” | `[0.82, 0.14]` |
| “estoy con temperatura” | `[0.79, 0.18]` |
| “quiero un turno” | `[0.10, 0.91]` |

Los dos primeros vectores son parecidos entre sí. En herramientas reales, el vector tiene muchas más coordenadas y la cercanía se puede medir con **similitud coseno**:

- cerca de `1`: orientaciones muy parecidas;
- cerca de `0`: poca relación en esa representación.

No hace falta calcularla a mano. La idea importante es: **convertimos textos en números para poder comparar su sentido aproximado**.

In [ ]:
# --- Demo breve: embeddings con sentence-transformers ---
import numpy as np

FAST_MODE = True
textos = [
    "Tengo dolor de cabeza",
    "Me duele mucho la cabeza",
    "Quiero pedir un turno",
]

if not FAST_MODE:
    textos.append("Necesito una cita con cardiología")
    textos.append("Quiero renovar una receta")

try:
    from sentence_transformers import SentenceTransformer

    modelo_embeddings = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2",
        device="cpu",
    )
    vectores = modelo_embeddings.encode(
        textos,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    similitudes = np.round(vectores @ vectores.T, 3)

    print("Forma de la matriz de embeddings:", vectores.shape)
    print("\nTextos:")
    for indice, texto in enumerate(textos):
        print(indice, "->", texto)
    print("\nSimilitud entre cada par:")
    print(similitudes)
except Exception as error:
    print("La demo de embeddings no pudo cargar el modelo.")
    print("El resto del notebook puede continuar sin esta descarga.")
    print("Detalle breve:", str(error)[:180])

### Qué observar en la matriz

- La diagonal vale `1` porque cada texto se compara consigo mismo.
- Las dos frases sobre dolor de cabeza deberían tener una similitud mayor entre sí.
- La frase sobre turnos debería quedar más alejada de las anteriores.

`sentence-transformers` ofrece modelos preparados para transformar una frase en un embedding con `encode()`. El modelo `all-MiniLM-L6-v2` es compacto y resulta práctico para experimentar en CPU.

> **Límite útil:** cercanía no garantiza comprensión perfecta. El embedding permite comparar y agrupar; siempre conviene mirar ejemplos concretos para entender qué considera parecido.

---
## 3. Clasificación: poner una etiqueta al texto completo

**Clasificar** es elegir una categoría para un texto. La entrada es el mensaje entero y la salida es una etiqueta de un conjunto conocido.

> **Analogía de las bandejas de recepción.** Llega un papel y lo colocás en una bandeja: `turno`, `receta` o `síntoma`. No estás copiando los datos del papel; estás decidiendo a qué bandeja pertenece.

```text
“¿Me dan cita con dermatología?”
                |
                v
          clasificador
      +---------+---------+
      |         |         |
    turno     receta    síntoma
      ^
      |
   etiqueta elegida
```

| Entrada | Etiqueta posible |
|---|---|
| “Quisiera una cita para el jueves” | `turno` |
| “Necesito renovar ibuprofeno” | `receta` |
| “Tengo fiebre desde ayer” | `síntoma` |

La etiqueta resume el propósito general. No indica todavía qué fecha, medicamento o síntoma apareció.

### Una herramienta simple: TF-IDF + clasificador

Con `scikit-learn` podés armar un primer clasificador en dos pasos:

```text
textos de ejemplo
      |
      v
TF-IDF: texto -> números según palabras útiles
      |
      v
clasificador: aprende fronteras entre etiquetas
      |
      v
etiqueta para un texto nuevo
```

**TF-IDF** aumenta el peso de términos que ayudan a distinguir mensajes y reduce el de palabras que aparecen en casi todos. Después, un clasificador simple relaciona esos números con las etiquetas de los ejemplos.

> **Analogía del resaltador.** Si en los mensajes de `turno` aparecen mucho “cita”, “día” y “horario”, TF-IDF ayuda a resaltarlas. El clasificador aprende a usar esas pistas, no una definición perfecta del idioma.

In [ ]:
# --- Demo breve: clasificar con scikit-learn ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

textos_entrenamiento = [
    "Quiero pedir un turno",
    "Necesito una cita para el martes",
    "¿Hay horario con cardiología?",
    "Necesito renovar una receta",
    "¿Me preparan la receta del medicamento?",
    "Quiero pedir mi medicación",
    "Tengo fiebre y tos",
    "Me duele la cabeza",
    "Siento dolor de garganta",
]
etiquetas_entrenamiento = [
    "turno", "turno", "turno",
    "receta", "receta", "receta",
    "síntoma", "síntoma", "síntoma",
]

clasificador = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2))),
    ("modelo", LogisticRegression(max_iter=500, random_state=42)),
])
clasificador.fit(textos_entrenamiento, etiquetas_entrenamiento)

mensajes_nuevos = [
    "Quisiera un horario para el viernes",
    "Tengo mucha tos",
    "Necesito otra receta",
]
predicciones = clasificador.predict(mensajes_nuevos)

for mensaje, etiqueta in zip(mensajes_nuevos, predicciones):
    print("Mensaje:", mensaje)
    print("Etiqueta elegida:", etiqueta)
    print("-" * 60)

### Dos caminos que conviene conocer

| Representación del texto | Clasificador | Qué aporta |
|---|---|---|
| TF-IDF | Regresión logística u otro modelo simple de `scikit-learn` | Es rápido, visible y funciona bien cuando las palabras dan buenas pistas |
| Embeddings | Clasificador simple de `scikit-learn` | Puede juntar expresiones parecidas aunque no usen exactamente las mismas palabras |

El segundo camino reemplaza TF-IDF por embeddings y conserva la idea general:

```text
texto -> embedding -> clasificador -> etiqueta
```

Para una primera exploración, TF-IDF permite ver el mecanismo con muy poco código. En ambos casos necesitás ejemplos con etiquetas para que el clasificador aprenda qué salida corresponde.

> **Importante:** esta demo tiene poquísimos ejemplos. Sirve para entender la herramienta, no para concluir que reconoce cualquier forma de hablar.

---
## 4. Extracción de entidades: sacar datos concretos

**Extraer entidades** significa localizar y organizar datos puntuales que aparecen dentro de un texto.

> **Analogía del formulario transparente.** Ponés un formulario arriba del mensaje y completás casilleros con lo que encontrás: medicamento, síntoma, especialidad y fecha. Lo que no aparece queda vacío.

```text
“Necesito dermatología el viernes por una erupción”
                   |
                   v
         extractor de entidades
                   |
      +------------+------------+
      |            |            |
      v            v            v
especialidad     fecha       síntoma
dermatología    viernes      erupción
```

| Tipo de entidad | Ejemplos sintéticos |
|---|---|
| Medicamento | ibuprofeno, amoxicilina |
| Síntoma | fiebre, tos, dolor de garganta |
| Especialidad | cardiología, pediatría |
| Fecha | martes, mañana, 12/10 |

### Clasificar no es extraer

Tomemos el mensaje: **“Necesito un turno con cardiología el martes”**.

| Operación | Pregunta | Salida |
|---|---|---|
| Clasificación | ¿De qué tipo es el mensaje completo? | `turno` |
| Extracción | ¿Qué datos concretos contiene? | `cardiología`, `martes` |

```text
                   mismo mensaje
                        |
             +----------+----------+
             |                     |
             v                     v
      una etiqueta            varios campos
         “turno”       especialidad, fecha, ...
```

Un sistema puede usar una sola operación o combinar ambas, según lo que necesite obtener.

### Primer método: listas controladas y reglas

Un **gazetteer** es una lista de términos conocidos. El extractor pasa por el mensaje y busca coincidencias. Para formatos regulares, como `12/10`, se puede sumar una expresión regular.

```text
mensaje en minúsculas
         |
   +-----+------------------+
   |                        |
   v                        v
buscar términos          aplicar regla
[fiebre, tos, ...]       día/mes
   |                        |
   +-----------+------------+
               v
         entidades halladas
```

| Ventaja | Límite |
|---|---|
| Es fácil ver por qué encontró algo | Solo reconoce formas previstas |
| Se corrige agregando o quitando términos | Sinónimos y errores de escritura se pueden escapar |
| No requiere un modelo pesado | Las coincidencias ambiguas necesitan más reglas |

Este método es ideal para aprender porque cada decisión queda a la vista.

In [ ]:
# --- Demo breve: una lista controlada encuentra términos exactos ---
mensaje_demo = "Tengo fiebre y tos desde el martes"
sintomas_demo = ["fiebre", "tos", "dolor de garganta"]

encontrados_demo = []
texto_demo = mensaje_demo.lower()
for sintoma in sintomas_demo:
    if sintoma in texto_demo:
        encontrados_demo.append(sintoma)

print("Mensaje:", mensaje_demo)
print("Síntomas encontrados:", encontrados_demo)

### Herramientas disponibles para extraer

| Enfoque | Herramienta | Cuándo resulta útil |
|---|---|---|
| Listas + reglas | Python básico y `re` | Vocabulario acotado, formatos previsibles y necesidad de inspeccionar cada regla |
| Modelo NER preparado | Pipeline de `transformers` | Textos más variados donde interesa detectar fragmentos aprendidos por un modelo |

**NER** significa reconocimiento de entidades nombradas. Un pipeline típico de `transformers` recibe texto y devuelve fragmentos con una categoría y un puntaje. La forma general es:

```python
# Esquema ilustrativo; no se ejecuta en esta clase.
# extractor = pipeline("token-classification", model="modelo-ner-elegido")
# extractor("texto a analizar")
```

Un modelo NER depende de las categorías y los ejemplos con los que fue preparado. Por eso, “usar NER” no garantiza que reconozca automáticamente medicamentos o síntomas en español. Primero hay que revisar qué entidades sabe detectar.

Para el laboratorio vamos a elegir el método más visible: **listas controladas + una regla de fecha**.

---
## Pausa de síntesis: cuatro operaciones, cuatro salidas

| Operación | Entra | Sale | Herramienta vista |
|---|---|---|---|
| Tokenización | Texto | Piezas + IDs | `tiktoken` |
| Embeddings | Texto | Vector numérico | `sentence-transformers` |
| Clasificación | Texto | Una etiqueta | TF-IDF + clasificador de `scikit-learn` |
| Extracción de entidades | Texto | Campos encontrados | Listas, reglas; también existen modelos NER en `transformers` |

```text
partir       representar       decidir categoría       localizar datos
tokens  ->   embedding    ->   clasificación      /    entidades
```

La práctica se concentra en la última operación para que puedas ver con claridad qué encuentra una regla y qué deja escapar.

---
## Mitad 2 — Práctica única: extractor simple para un consultorio

## El caso

Vas a trabajar con nueve mensajes **sintéticos e inventados**. El objetivo es sacar cuatro tipos de datos:

- medicamentos;
- especialidades;
- síntomas;
- fechas.

El extractor será deliberadamente simple:

```text
mensajes inventados
        |
        v
pasar a minúsculas
        |
   +----+-------------------+
   |                        |
   v                        v
buscar listas          buscar fecha
controladas            con una regla
   |                        |
   +------------+-----------+
                v
          tabla de resultados
```

Primero ejecutá todo sin cambiar nada. Después modificá **solo** las listas marcadas con `TODO`, volvé a ejecutar y compará.

## Paso 1 — Conocer los mensajes y las listas iniciales

Las listas arrancan incompletas a propósito. Algunas entidades se encontrarán y otras se escaparán.

> **Tu trabajo no es escribir un modelo complejo.** Tu trabajo es observar, formular una regla simple y comprobar qué cambia.

In [ ]:
# --- Datos sintéticos del laboratorio ---
mensajes_consultorio = [
    "Necesito cardiología para el martes. Tengo dolor de pecho.",
    "¿Me renuevan la receta de ibuprofeno? Me duele la cabeza.",
    "Quiero dermatología el 12/10 por una erupción.",
    "Desde ayer tengo fiebre y tos.",
    "Turno con pediatría para el viernes.",
    "Estoy tomando amoxicilina y tengo náuseas.",
    "Necesito clínica médica mañana por dolor de garganta.",
    "Consulta con traumatología: me torcí el tobillo.",
    "Tomo paracetamol y tengo dolor muscular desde el jueves.",
]

# TODO 1: agregá medicamentos que veas en los mensajes y falten acá.
MEDICAMENTOS = [
    "ibuprofeno",
    "amoxicilina",
]

# TODO 2: agregá especialidades que veas en los mensajes y falten acá.
ESPECIALIDADES = [
    "cardiología",
    "dermatología",
    "pediatría",
]

# TODO 3: agregá síntomas o frases de síntomas que falten acá.
SINTOMAS = [
    "fiebre",
    "tos",
    "dolor de garganta",
]

# Esta lista ya está completa para los ejemplos del laboratorio.
FECHAS_RELATIVAS = [
    "ayer", "hoy", "mañana",
    "lunes", "martes", "miércoles", "jueves",
    "viernes", "sábado", "domingo",
]

print("Mensajes cargados:", len(mensajes_consultorio))
print("Medicamentos iniciales:", MEDICAMENTOS)
print("Especialidades iniciales:", ESPECIALIDADES)
print("Síntomas iniciales:", SINTOMAS)

## Paso 2 — Leer el extractor antes de ejecutarlo

La función `buscar_terminos` recorre una lista y conserva los términos presentes en el mensaje. `extraer_fecha` hace dos búsquedas:

1. palabras conocidas, como `martes` o `mañana`;
2. un patrón numérico sencillo, como `12/10`.

```text
“dermatología el 12/10 por una erupción”
          |          |           |
          v          v           v
   lista de         regla       lista de
   especialidades   numérica    síntomas
```

Si un término no está en la lista con una forma compatible, no aparecerá en la salida. Esa limitación es justamente lo que vas a observar.

In [ ]:
# --- Funciones del extractor: listas + una regla de fecha ---
import re

def buscar_terminos(texto, lista_controlada):
    encontrados = []
    texto_minusculas = texto.lower()

    for termino in lista_controlada:
        if termino.lower() in texto_minusculas:
            encontrados.append(termino)

    return encontrados


def extraer_fechas(texto):
    fechas = buscar_terminos(texto, FECHAS_RELATIVAS)

    # Reconoce día/mes, por ejemplo 12/10.
    fechas_numericas = re.findall(r"\b\d{1,2}/\d{1,2}\b", texto)
    for fecha in fechas_numericas:
        if fecha not in fechas:
            fechas.append(fecha)

    return fechas


def extraer_entidades(texto):
    entidades = {
        "medicamentos": buscar_terminos(texto, MEDICAMENTOS),
        "especialidades": buscar_terminos(texto, ESPECIALIDADES),
        "síntomas": buscar_terminos(texto, SINTOMAS),
        "fechas": extraer_fechas(texto),
    }
    return entidades


print("Extractor listo.")

## Paso 3 — Ejecutar y mirar la tabla

No mires solamente las filas que funcionan. Prestá atención a los casilleros vacíos:

- ¿el dato no estaba en el mensaje?;
- ¿o estaba escrito, pero faltaba en la lista controlada?

Esa diferencia permite separar una **ausencia real** de una **limitación de nuestra regla**.

In [ ]:
# --- Aplicar el extractor a todos los mensajes ---
import pandas as pd

filas_resultado = []

for numero, mensaje in enumerate(mensajes_consultorio, start=1):
    entidades = extraer_entidades(mensaje)
    fila = {
        "id": numero,
        "mensaje": mensaje,
        "medicamentos": ", ".join(entidades["medicamentos"]),
        "especialidades": ", ".join(entidades["especialidades"]),
        "síntomas": ", ".join(entidades["síntomas"]),
        "fechas": ", ".join(entidades["fechas"]),
    }
    filas_resultado.append(fila)

tabla_resultados = pd.DataFrame(filas_resultado)
pd.set_option("display.max_colwidth", 80)
tabla_resultados

## Paso 4 — Comparar con una referencia visible

La referencia de abajo no representa una verdad médica. Solo indica qué fragmentos decidimos reconocer en este ejercicio.

| Mensaje | Medicamento | Especialidad | Síntoma | Fecha |
|---:|---|---|---|---|
| 1 | — | cardiología | dolor de pecho | martes |
| 2 | ibuprofeno | — | me duele la cabeza | — |
| 3 | — | dermatología | erupción | 12/10 |
| 4 | — | — | fiebre, tos | ayer |
| 5 | — | pediatría | — | viernes |
| 6 | amoxicilina | — | náuseas | — |
| 7 | — | clínica médica | dolor de garganta | mañana |
| 8 | — | traumatología | torcí el tobillo | — |
| 9 | paracetamol | — | dolor muscular | jueves |

Ejecutá la celda siguiente. Te mostrará, mensaje por mensaje, qué elementos de esa referencia todavía no encontró tu extractor.

In [ ]:
# --- Referencia didáctica para hacer visibles los huecos ---
referencia_didactica = [
    {"medicamentos": [], "especialidades": ["cardiología"], "síntomas": ["dolor de pecho"], "fechas": ["martes"]},
    {"medicamentos": ["ibuprofeno"], "especialidades": [], "síntomas": ["me duele la cabeza"], "fechas": []},
    {"medicamentos": [], "especialidades": ["dermatología"], "síntomas": ["erupción"], "fechas": ["12/10"]},
    {"medicamentos": [], "especialidades": [], "síntomas": ["fiebre", "tos"], "fechas": ["ayer"]},
    {"medicamentos": [], "especialidades": ["pediatría"], "síntomas": [], "fechas": ["viernes"]},
    {"medicamentos": ["amoxicilina"], "especialidades": [], "síntomas": ["náuseas"], "fechas": []},
    {"medicamentos": [], "especialidades": ["clínica médica"], "síntomas": ["dolor de garganta"], "fechas": ["mañana"]},
    {"medicamentos": [], "especialidades": ["traumatología"], "síntomas": ["torcí el tobillo"], "fechas": []},
    {"medicamentos": ["paracetamol"], "especialidades": [], "síntomas": ["dolor muscular"], "fechas": ["jueves"]},
]

tipos_entidad = ["medicamentos", "especialidades", "síntomas", "fechas"]

for indice, mensaje in enumerate(mensajes_consultorio):
    obtenido = extraer_entidades(mensaje)
    esperado = referencia_didactica[indice]
    faltantes = {}

    for tipo in tipos_entidad:
        faltantes[tipo] = []
        for valor in esperado[tipo]:
            if valor not in obtenido[tipo]:
                faltantes[tipo].append(valor)

    print("Mensaje", indice + 1)
    print("Obtenido:", obtenido)
    print("Todavía falta:", faltantes)
    print("-" * 80)

## Paso 5 — Modificar solamente los `TODO`

Volvé a la celda del Paso 1 y completá las tres listas. Después ejecutá otra vez desde esa celda hasta la comparación.

Preguntas para guiar tu prueba:

1. ¿Qué palabra o frase aparece literalmente en el mensaje y falta en una lista?
2. ¿A qué tipo de entidad pertenece?
3. Después de agregarla, ¿desaparece del bloque `Todavía falta`?
4. ¿Agregar un término provocó alguna coincidencia que no esperabas?

### Solución de referencia

Si necesitás contrastar tu trabajo, estas son las incorporaciones mínimas para este conjunto:

```python
# MEDICAMENTOS: agregar "paracetamol"
# ESPECIALIDADES: agregar "clínica médica" y "traumatología"
# SINTOMAS: agregar "dolor de pecho", "me duele la cabeza",
#           "erupción", "náuseas", "torcí el tobillo" y "dolor muscular"
```

La solución funciona porque copia formas presentes en estos nueve mensajes. Si alguien escribe “cefalea” en lugar de “me duele la cabeza”, la lista volverá a quedar corta. Esa fragilidad no es un error oculto: es el límite natural de una coincidencia exacta.

## Qué aprendiste al tocar las listas

| Lo que observaste | Idea que demuestra |
|---|---|
| Un término agregado empieza a aparecer | El comportamiento depende de reglas visibles |
| Un sinónimo no previsto queda afuera | Una lista no comprende todas las formas de decir algo |
| `12/10` aparece sin estar en la lista | Una regla puede reconocer un formato |
| Un mensaje puede devolver varios campos | Extraer no es elegir una sola etiqueta |

> **Resultado del laboratorio:** construiste un extractor pequeño, entendiste cada decisión y pudiste señalar exactamente por qué encontró o perdió una entidad.

---
## Síntesis de la clase

- Un modelo no lee el texto como nosotros: el tokenizador lo parte en **tokens** y los convierte en IDs.
- Un **embedding** es una representación numérica aproximada de “de qué se trata” un texto; textos parecidos suelen quedar cerca.
- **Clasificar** es asignar una etiqueta al texto completo. TF-IDF y los clasificadores de `scikit-learn` ofrecen un punto de partida simple.
- **Extraer entidades** es localizar datos concretos dentro del texto, como un medicamento, un síntoma, una especialidad o una fecha.
- Las listas y reglas son fáciles de inspeccionar, pero se les escapan formas no previstas.
- También existen modelos NER accesibles desde `transformers`; antes de elegir uno hay que saber qué categorías reconoce.

```text
texto
  +-> tokens: ¿en qué piezas se parte?
  +-> embedding: ¿dónde queda en el mapa de significados?
  +-> clasificación: ¿qué etiqueta recibe?
  +-> entidades: ¿qué datos concretos contiene?
```

## Comprobación conceptual

Respondé con tus palabras, sin copiar definiciones:

1. ¿Por qué un token no es necesariamente una palabra? Inventá un ejemplo posible de subpalabra.
2. Si dos mensajes tienen embeddings cercanos, ¿qué interpretación útil podés hacer y qué no podés asegurar?
3. Para “Necesito pediatría el viernes”, ¿qué devolvería una clasificación y qué devolvería una extracción de entidades?
4. ¿Qué ventaja y qué límite tiene resolver entidades con una lista controlada?

Si podés explicar las cuatro respuestas usando las salidas que viste, alcanzaste el objetivo de esta clase.